# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Machine Learning: Alternating Least Squares (ALS)** </center>
---
<center>Saul Razo Magallanes - 739974</center>
    <center>Ingeniería en Sistemas Computacionales</center>
    <center><strong>Profesor:</strong> Pablo Camarillo Ramírez</center>
    <center><strong>Fecha:</strong> 22/04/2026</center>

# Create SparkSession

In [24]:
from pcamarillor.spark_utils import SparkUtils

su = SparkUtils("ML: ALS", 
                "spark://spark-master:7077")
su.spark

# Example 1: Songs recommednation

In [2]:
# Sample user-song interaction data
data = [(1, 1, 4),
        (1, 2, 5),
        (1, 5, 5),
        (2, 2, 3),
        (2, 3, 4),
        (2, 4, 3),
        (3, 1, 2),
        (3, 3, 5),
        (3, 5, 1)]
  
# Define schema for the DataFrame
schema = SparkUtils.generate_schema([("user_id", "int"), ("song_id", "int"), ("rating", "int")])

# Create DataFrame for interactions
interactions_df = su.spark.createDataFrame(data, schema)
interactions_df.show()

[Stage 0:>                                                          (0 + 1) / 1]

+-------+-------+------+
|user_id|song_id|rating|
+-------+-------+------+
|      1|      1|     4|
|      1|      2|     5|
|      1|      5|     5|
|      2|      2|     3|
|      2|      3|     4|
|      2|      4|     3|
|      3|      1|     2|
|      3|      3|     5|
|      3|      5|     1|
+-------+-------+------+



In [3]:
print(f"Number of items o canciones (n):{interactions_df.groupBy('song_id').count().count()}")
print(f"Number of users (m):{interactions_df.groupBy('user_id').count().count()}")

Number of items o canciones (n):5
Number of users (m):3


In [8]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="user_id", 
    itemCol="song_id", 
    ratingCol="rating", 
    maxIter=10, 
    regParam=0.1, 
    rank=5, # Controls the dimensionality of the latent vector space for 
            # users and items.
    coldStartStrategy="drop"  # Avoids NaN predictions
)

In [9]:
model = als.fit(interactions_df)
print("Recommendation system generated successfully")

Recommendation system generated successfully


In [6]:
# Generate recommendations for each user
user_recommendations = model.recommendForAllUsers(numItems=3)

# Show recommendations
user_recommendations.show(truncate=False)

[Stage 91:==========================================>            (77 + 1) / 100]

+-------+------------------------------------------------+
|user_id|recommendations                                 |
+-------+------------------------------------------------+
|1      |[{2, 4.955084}, {5, 4.858411}, {1, 3.9437413}]  |
|2      |[{3, 3.9474297}, {2, 2.9692369}, {4, 2.9077086}]|
|3      |[{3, 4.8387585}, {4, 3.1941473}, {2, 2.3443584}]|
+-------+------------------------------------------------+



In [14]:
songs = [
    (1, "song a"),
    (2, "song b"),
    (3, "song c"),
    (4, "song d"),
    (5, "song e")]

songs_schema = SparkUtils.generate_schema([("song_id", "int"), ("title", "string")])
songs_df = su.spark.createDataFrame(songs, songs_schema)

In [11]:
from pyspark.sql.functions import explode

# Explode recommendations for easier reading
recommendations = user_recommendations.select("user_id", explode("recommendations").alias("rec"))
recommendations = recommendations.join(songs_df, recommendations.rec.song_id == songs_df.song_id).select("user_id", "title", "rec.rating")

# Show user-song recommendations with titles
recommendations.show(truncate=False)

[Stage 195:=============>(99 + 1) / 100][Stage 197:>                (0 + 1) / 1]

+-------+------+---------+
|user_id|title |rating   |
+-------+------+---------+
|1      |song b|4.955084 |
|1      |song e|4.858411 |
|1      |song a|3.9437413|
|2      |song c|3.9474297|
|2      |song b|2.9692369|
|2      |song d|2.9077086|
|3      |song c|4.8387585|
|3      |song d|3.1941473|
|3      |song b|2.3443584|
+-------+------+---------+



In [12]:
predictions = model.transform(interactions_df)
predictions.show(truncate=False)

+-------+-------+------+----------+
|user_id|song_id|rating|prediction|
+-------+-------+------+----------+
|1      |1      |4     |3.9437413 |
|1      |2      |5     |4.955084  |
|1      |5      |5     |4.858411  |
|2      |2      |3     |2.9692369 |
|3      |1      |2     |1.9652493 |
|3      |3      |5     |4.8387585 |
|3      |5      |1     |1.0472596 |
|2      |3      |4     |3.9474297 |
|2      |4      |3     |2.9077086 |
+-------+-------+------+----------+



In [13]:
# Evaluate the Recommendation System
from pyspark.ml.evaluation import RegressionEvaluator
# Set up evaluator to compute RMSE
evaluator = RegressionEvaluator(
    metricName="rmse", 
    labelCol="rating", 
    predictionCol="prediction"
)

# Calculate RMSE
rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error (RMSE) = {rmse}")

Root-mean-square error (RMSE) = 0.08621523923223268


# Lab 12: Building a Recommendation System with ALS 

In [25]:
movies_ratings_path = "/opt/spark/work-dir/data/ml/als"

movies_ratings_schema = SparkUtils.generate_schema([("user_id", "int"), ("movie_id", "int"), ("rating", "int"),("timestamp", "int")])

# Source https://github.com/databricks/Spark-The-Definitive-Guide/blob/master/data/sample_movielens_ratings.txt
movies_ratings_df = su.spark.read \
                    .option("header", "false") \
                    .option("delimiter", "::") \
                    .schema(movies_ratings_schema) \
                    .csv(movies_ratings_path)

movies_ratings_df.printSchema()
movies_ratings_df.show(n=3)

root
 |-- user_id: integer (nullable = true)
 |-- movie_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- timestamp: integer (nullable = true)

+-------+--------+------+----------+
|user_id|movie_id|rating| timestamp|
+-------+--------+------+----------+
|      0|       2|     3|1424380312|
|      0|       3|     1|1424380312|
|      0|       5|     2|1424380312|
+-------+--------+------+----------+
only showing top 3 rows


## Create & Train the ML Model

In [26]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="user_id", 
    itemCol="movie_id", 
    ratingCol="rating", 
    maxIter=10, 
    regParam=0.1, 
    rank=5, # Controls the dimensionality of the latent vector space for 
            # users and items.
    coldStartStrategy="drop"  # Avoids NaN predictions
)

In [27]:
model = als.fit(movies_ratings_df)
print("Recommendation system generated successfully")

Recommendation system generated successfully


In [28]:
# Generate recommendations for each user
user_recommendations = model.recommendForAllUsers(numItems=3)

# Show recommendations
user_recommendations.show(truncate=False)

[Stage 634:===========================================>          (80 + 1) / 100]

+-------+---------------------------------------------------+
|user_id|recommendations                                    |
+-------+---------------------------------------------------+
|0      |[{92, 2.6710804}, {2, 2.5510902}, {93, 2.4591002}] |
|10     |[{92, 2.9545257}, {93, 2.9320107}, {2, 2.9095814}] |
|20     |[{22, 3.2966967}, {75, 3.072588}, {77, 3.0639844}] |
|1      |[{22, 2.633828}, {77, 2.5806947}, {53, 2.4940863}] |
|11     |[{46, 4.4955196}, {27, 4.4739137}, {18, 4.3025823}]|
|21     |[{29, 4.201056}, {52, 4.098069}, {62, 3.6557717}]  |
|22     |[{75, 4.4546614}, {51, 4.369368}, {77, 4.1656}]    |
|2      |[{93, 4.344805}, {83, 4.1517253}, {8, 4.128674}]   |
|12     |[{46, 5.786538}, {55, 4.6762357}, {32, 4.216643}]  |
|23     |[{46, 5.9483185}, {55, 4.8195815}, {32, 4.746619}] |
|3      |[{51, 4.0655594}, {69, 3.9054694}, {75, 3.8363006}]|
|13     |[{93, 2.6741734}, {29, 2.459445}, {52, 2.4313002}] |
|24     |[{29, 4.4677954}, {52, 4.4239097}, {85, 4.192702}] |
|4      

## Persist the model

In [31]:
model_path = "/opt/spark/work-dir/models/als_movies"
model.save(model_path)
print(f"Model saved to {model_path}")

from pyspark.ml.recommendation import ALSModel
loaded_model = ALSModel.load(model_path)
print("Model loaded successfully")

Py4JJavaError: An error occurred while calling o704.save.
: java.io.IOException: Path /opt/spark/work-dir/models/als_movies already exists. To overwrite it, please use write.overwrite().save(path) for Scala and use write().overwrite().save(path) for Java and Python.
	at org.apache.spark.ml.util.FileSystemOverwrite.handleOverwrite(ReadWrite.scala:794)
	at org.apache.spark.ml.util.MLWriter.save(ReadWrite.scala:168)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:77)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:569)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
	at py4j.Gateway.invoke(Gateway.java:282)
	at py4j.commands.AbstractCommand.invokeMethod(Ab

## Predictions

In [32]:
from pyspark.sql.functions import explode

# Explode recommendations for easier reading
recommendations = user_recommendations.select("user_id", explode("recommendations").alias("rec"))
recommendations = recommendations.select("user_id", "rec.movie_id", "rec.rating")

# Show user-movie recommendations
recommendations.show(truncate=False)

predictions = model.transform(movies_ratings_df)
predictions.show(truncate=False)

+-------+--------+---------+
|user_id|movie_id|rating   |
+-------+--------+---------+
|0      |92      |2.6710804|
|0      |2       |2.5510902|
|0      |93      |2.4591002|
|10     |92      |2.9545257|
|10     |93      |2.9320107|
|10     |2       |2.9095814|
|20     |22      |3.2966967|
|20     |75      |3.072588 |
|20     |77      |3.0639844|
|1      |22      |2.633828 |
|1      |77      |2.5806947|
|1      |53      |2.4940863|
|11     |46      |4.4955196|
|11     |27      |4.4739137|
|11     |18      |4.3025823|
|21     |29      |4.201056 |
|21     |52      |4.098069 |
|21     |62      |3.6557717|
|22     |75      |4.4546614|
|22     |51      |4.369368 |
+-------+--------+---------+
only showing top 20 rows
+-------+--------+------+----------+----------+
|user_id|movie_id|rating|timestamp |prediction|
+-------+--------+------+----------+----------+
|22     |0       |1     |1424380312|1.0024675 |
|22     |3       |2     |1424380312|1.6375351 |
|22     |5       |2     |1424380312|2.1

## Test ML Model

In [33]:
# Evaluate the Recommendation System
from pyspark.ml.evaluation import RegressionEvaluator

# Set up evaluator to compute RMSE
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

# Calculate RMSE
rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error (RMSE) = {rmse}")

Root-mean-square error (RMSE) = 0.5792750429679359


In [ ]:
su.spark.stop()